# Day 3 Practice — Handling Inappropriate Data

**Dataset:** `inappropriate_data_dataset_50_records.csv`

In [1]:
import pandas as pd
import numpy as np

In [7]:
data = pd.read_csv("inappropriate_data_dataset_50_records_1.csv")
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52 entries, 0 to 51
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Customer_ID      52 non-null     int64  
 1   Age              52 non-null     float64
 2   Monthly_Income   52 non-null     float64
 3   Credit_Score     52 non-null     float64
 4   Gender           52 non-null     object 
 5   Education_Level  52 non-null     object 
 6   City             52 non-null     object 
 7   Loan_Status      52 non-null     object 
 8   City_dup         52 non-null     object 
dtypes: float64(3), int64(1), object(5)
memory usage: 3.8+ KB


In [8]:
data.head()

,Customer_ID,Age,Monthly_Income,Credit_Score,Gender,Education_Level,City,Loan_Status,City_dup
0,1008,83.0,78578.27,488.0,FEMALE,Gradute,PUNE,Pending,PUNE
1,1077,32.0,66351.58,840.0,Femlae,PG,Pune,rejected,Pune
2,1065,47.0,70782.39,571.0,FEMALE,Undergraduate,Delhi,approved,Delhi
3,1043,-5.0,77625.76,704.0,male,Under Graduate,Hyderabad,Pending,Hyderabad
4,1043,59.0,35856.88,547.0,M,post graduate,Chennai,Rejectd,Chennai


Q1. Column Type Identification

- Customer_ID -------> Numerical (Discrete) — identifier, but technically discrete
- Age -------> Numerical (Continuous) — can take fractional/decimal values in real life (though our domain guideline may say otherwise, decided in Q4)
- Monthly_Income` -------> Numerical (Continuous)
- Credit_Score -------> Numerical (Discrete) — credit scores are whole numbers by domain definition
- Gender -------> Categorical
- Education_Level -------> Categorical (could be Ordinal if ranked UG < PG < PhD, but treated as Categorical here per class convention)
- City / City_dup -------> Categorical
- Loan_Status -------> Categorical

In [12]:
data.dtypes

Customer_ID          int64
Age                float64
Monthly_Income     float64
Credit_Score       float64
Gender              object
Education_Level     object
City                object
Loan_Status         object
City_dup            object
dtype: object

 Q2. Duplicate Records

Check for and remove fully duplicate rows.

In [15]:
print('Duplicate rows:', data.duplicated().sum())

data.drop_duplicates(inplace=True, ignore_index=True)

print(' After Removing :', data.shape)

Duplicate rows: 0
 After Removing : (50, 9)


Q3. Duplicate Columns

City and City_dup look identical.

In [17]:
data[data['City']!=data['City_dup']]

,Customer_ID,Age,Monthly_Income,Credit_Score,Gender,Education_Level,City,Loan_Status,City_dup


 Q4. Numerical Column — `Age`

Domain decisions:
- Negative values -> NOT allowed (delete/NaN)
- Decimals -> NOT allowed for age in this domain (age is reported in whole years)
- Valid range -> 18 to 100 (assume domain expert says minimum loan-eligible age is 18, and 100 is a realistic human upper bound)

In [22]:
# Check violations before fixing
print('Negative ages:\n', data[data['Age'] < 0][['Customer_ID','Age']])
print('\nDecimal ages:\n', data[data['Age'] % 1 != 0][['Customer_ID','Age']])
print('\nOut-of-range ages (<18 or >100):\n', data[(data['Age'] < 18) | (data['Age'] > 100)][['Customer_ID','Age']])

Negative ages:
    Customer_ID  Age
3         1043 -5.0

Decimal ages:
     Customer_ID   Age
8          1020  29.5
31         1022  45.2

Out-of-range ages (<18 or >100):
     Customer_ID    Age
3          1043   -5.0
12         1073   17.0
13         1076  102.0
17         1012  150.0


In [23]:
invalid_age = (data['Age'] < 0) | (data['Age'] % 1 != 0) | (data['Age'] < 18) | (data['Age'] > 100)
data.loc[invalid_age, 'Age'] = np.nan

print('Remaining valid Age count:', data['Age'].notna().sum())

Remaining valid Age count: 44


Q5. Numerical Column — `Monthly_Income`

Domain decisions:
- Negative income -> NOT allowed (delete/NaN)
- Zero income -> treated as inappropriate for a *loan applicant* dataset (someone with literally 0 income being evaluated for a loan is a data-entry issue, not a valid case) -> NaN
- Decimals -> allowed (income is naturally fractional, e.g. 78578.27)

In [24]:
data[data['Monthly_Income'] < 0][['Customer_ID','Monthly_Income']]
data[data['Monthly_Income'] == 0][['Customer_ID','Monthly_Income']]

,Customer_ID,Monthly_Income
22,1018,0.0


In [25]:
invalid_income = (data['Monthly_Income'] <= 0)
data.loc[invalid_income, 'Monthly_Income'] = np.nan

print('Remaining valid Monthly_Income count:', data['Monthly_Income'].notna().sum())

Remaining valid Monthly_Income count: 48


Q6. Numerical Column — `Credit_Score`

Domain decisions:
- Valid range -> 300 to 900
- Decimals -> NOT allowed (credit scores are always whole numbers)

In [26]:
data[(data['Credit_Score'] < 300) | (data['Credit_Score'] > 900)][['Customer_ID','Credit_Score']]
data[data['Credit_Score'] % 1 != 0][['Customer_ID','Credit_Score']]

,Customer_ID,Credit_Score
9,1009,720.5


In [27]:
invalid_score = (data['Credit_Score'] < 300) | (data['Credit_Score'] > 900) | (data['Credit_Score'] % 1 != 0)
data.loc[invalid_score, 'Credit_Score'] = np.nan

print('Remaining valid Credit_Score count:', data['Credit_Score'].notna().sum())

Remaining valid Credit_Score count: 46


Q7. Categorical Column — Gender

Normalize spelling, casing, and whitespace errors.

In [28]:
data['Gender'].unique()

array(['FEMALE', 'Femlae', 'male', 'M', 'Unknown', 'F', 'female',
       'femAle', 'Male', ' MALE '], dtype=object)

In [29]:
gender_map = {
    'FEMALE': 'Female', 'female': 'Female', 'Femlae': 'Female', 'femAle': 'Female', 'F': 'Female',
    'MALE': 'Male', 'male': 'Male', 'M': 'Male', ' MALE ': 'Male'
}

data['Gender'] = data['Gender'].str.strip().replace(gender_map)
data['Gender'].unique()

array(['Female', 'Male', 'Unknown'], dtype=object)

In [30]:
# 'Unknown' is not a real gender category -> treat those specific records as inappropriate
print('Unknown gender rows:', (data['Gender'] == 'Unknown').sum())
data = data[data['Gender'] != 'Unknown'].reset_index(drop=True)
data['Gender'].unique()

Unknown gender rows: 7


array(['Female', 'Male'], dtype=object)

Q8. Categorical Column — Education_Level

Unify UG and PG variants into single labels.

In [31]:
data['Education_Level'].unique()

array(['Gradute', 'PG', 'Undergraduate', 'Under Graduate',
       'post graduate', 'Graduate', 'Post Graduate', 'PhD', 'UG'],
      dtype=object)

In [32]:
edu_map = {
    'Gradute': 'Graduate', 'Graduate': 'Graduate',
    'Under Graduate': 'Undergraduate', 'Undergraduate': 'Undergraduate', 'UG': 'Undergraduate',
    'Post Graduate': 'Post Graduate', 'post graduate': 'Post Graduate', 'PG': 'Post Graduate',
    'PhD': 'PhD'
}

data['Education_Level'] = data['Education_Level'].str.strip().replace(edu_map)
data['Education_Level'].unique()

array(['Graduate', 'Post Graduate', 'Undergraduate', 'PhD'], dtype=object)

Q9. Categorical Column — City

Fix case + whitespace, and unify Bangalore/Bengaluru.

In [33]:
data['City'].unique()

array(['PUNE', 'Pune', 'Delhi', 'Hyderabad', 'Chennai', 'Mumbai',
       'Bengaluru', 'Bangalore', 'mumbai ', ' Delhi '], dtype=object)

In [34]:
data['City'] = data['City'].str.strip().str.title()

city_map = {'Bangalore': 'Bengaluru'}   # unify to one official city name
data['City'] = data['City'].replace(city_map)
data['City'].unique()

array(['Pune', 'Delhi', 'Hyderabad', 'Chennai', 'Mumbai', 'Bengaluru'],
      dtype=object)

Q10. Categorical Column — Loan_Status

Unify approved/rejected/pending variants.

In [35]:
data['Loan_Status'].unique()

array(['Pending', 'rejected ', 'approved', 'Rejectd', 'Rejected',
       'Approved', 'APPROVED'], dtype=object)

In [36]:
status_map = {
    'Rejectd': 'Rejected', 'rejected': 'Rejected', 'Rejected': 'Rejected', 'REJECTED': 'Rejected',
    'approved': 'Approved', 'Approved': 'Approved', 'APPROVED': 'Approved',
    'Pending': 'Pending'
}

data['Loan_Status'] = data['Loan_Status'].str.strip().replace(status_map)
data['Loan_Status'].unique()

array(['Pending', 'Rejected', 'Approved'], dtype=object)

Q11. Post-Cleaning Verification

In [37]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43 entries, 0 to 42
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Customer_ID      43 non-null     int64  
 1   Age              37 non-null     float64
 2   Monthly_Income   41 non-null     float64
 3   Credit_Score     40 non-null     float64
 4   Gender           43 non-null     object 
 5   Education_Level  43 non-null     object 
 6   City             43 non-null     object 
 7   Loan_Status      43 non-null     object 
dtypes: float64(3), int64(1), object(4)
memory usage: 2.8+ KB


In [38]:
data.describe()

,Customer_ID,Age,Monthly_Income,Credit_Score
count,43.000000,37.000000,41.000000,40.000000
mean,1053.651163,59.459459,62616.907561,592.975000
std,27.843853,25.253817,18228.410391,163.220079
min,1006.000000,18.000000,31253.310000,301.000000
25%,1036.000000,35.000000,51761.480000,478.000000
50%,1055.000000,59.000000,61173.910000,615.500000
75%,1077.000000,82.000000,74943.210000,721.250000
max,1097.000000,99.000000,125000.750000,879.000000


**Summary of changes:**
- Removed exact duplicate rows and the redundant City_dup column
- Age, Monthly_Income, and Credit_Score had invalid entries (negative, decimal-where-not-allowed, out-of-range) converted to NaN rather than dropping full rows
- Gender, Education_Level, City, and Loan_Status had spelling, casing, and whitespace inconsistencies normalized into single canonical categories
- Rows with Gender == Unknown were removed since it isn't a valid category for this domain